In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

In [29]:
df=pd.read_csv("Ensemble_learning_datasets.csv")

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      1983 non-null   float64
 1   Annual_Income            1979 non-null   float64
 2   Credit_Score             1976 non-null   float64
 3   Employment_Years         1981 non-null   float64
 4   Debt_to_Income_Ratio     1981 non-null   float64
 5   Loan_Amount              1982 non-null   float64
 6   Interest_Rate            1986 non-null   float64
 7   Monthly_Expenses         1980 non-null   float64
 8   Savings_Balance          1982 non-null   float64
 9   Late_Payments_Last_Year  1979 non-null   float64
 10  Existing_Loans_Count     1980 non-null   float64
 11  Education_Level          1980 non-null   object 
 12  Employment_Type          1981 non-null   object 
 13  Home_Ownership           1972 non-null   object 
 14  Marital_Status          

In [31]:
df.isnull().sum()

Age                        17
Annual_Income              21
Credit_Score               24
Employment_Years           19
Debt_to_Income_Ratio       19
Loan_Amount                18
Interest_Rate              14
Monthly_Expenses           20
Savings_Balance            18
Late_Payments_Last_Year    21
Existing_Loans_Count       20
Education_Level            20
Employment_Type            19
Home_Ownership             28
Marital_Status             22
Loan_Status                 0
dtype: int64

In [32]:
df.describe()

,Age,Annual_Income,Credit_Score,Employment_Years,Debt_to_Income_Ratio,Loan_Amount,Interest_Rate,Monthly_Expenses,Savings_Balance,Late_Payments_Last_Year,Existing_Loans_Count
count,1983.00000,1979.000000,1976.000000,1981.000000,1981.000000,1982.000000,1986.000000,1980.000000,1982.000000,1979.000000,1980.000000
mean,44.03530,65373.123441,679.485117,7.565456,0.304657,18575.679303,10.675917,2707.538083,17782.719588,0.712986,2.039394
std,15.69647,34130.920935,59.169226,4.776487,0.169227,9795.724341,3.329399,1626.881822,27295.207226,1.075740,1.519110
min,-5.00000,-5000.000000,-100.000000,-4.000000,-0.500000,-10000.000000,-8.000000,-2000.000000,-50000.000000,-3.000000,-2.000000
25%,31.00000,49702.549771,642.574947,4.176171,0.178502,12656.442076,8.505406,1707.494603,7397.550734,0.000000,1.000000
50%,44.00000,65158.555328,679.036625,6.557815,0.286143,18560.364657,10.568656,2530.691185,12818.760698,0.000000,2.000000
75%,57.00000,79413.053486,714.496312,10.018354,0.406848,24084.769077,12.758860,3511.219534,21045.875000,1.000000,3.000000
max,150.00000,1000000.000000,1200.000000,60.000000,2.500000,250000.000000,65.000000,50000.000000,1000000.000000,30.000000,25.000000


In [33]:
numeric_features=df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features=df.select_dtypes(include=[object]).columns.tolist()

In [34]:
for num in numeric_features:
    median_value=df[num].median()
    df[num].fillna(median_value,inplace=True)

C:\Users\vaish\AppData\Local\Temp\ipykernel_25372\2948291940.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[num].fillna(median_value,inplace=True)


In [35]:
for num in categorical_features:
    mode_values=df[num].mode()[0]
    df[num].fillna(mode_values,inplace=True)

C:\Users\vaish\AppData\Local\Temp\ipykernel_25372\2580118123.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[num].fillna(mode_values,inplace=True)


In [36]:
df.isnull().sum()

Age                        0
Annual_Income              0
Credit_Score               0
Employment_Years           0
Debt_to_Income_Ratio       0
Loan_Amount                0
Interest_Rate              0
Monthly_Expenses           0
Savings_Balance            0
Late_Payments_Last_Year    0
Existing_Loans_Count       0
Education_Level            0
Employment_Type            0
Home_Ownership             0
Marital_Status             0
Loan_Status                0
dtype: int64

In [37]:
# handel the outliers

for num in numeric_features:
    q1=df[num].quantile(0.25)
    q3=df[num].quantile(0.75)

    iqr=q3-q1

    lb=q1-1.5*iqr
    ub=q3+1.5*iqr

    df[num]=np.clip(df[num],lb,ub)

In [38]:
df["Marital_Status"].unique()


array(['Single', 'Married', 'Divorced'], dtype=object)

In [39]:
df["Education_Level"].unique()


array(['Bachelor', 'High School', 'Diploma', 'Master'], dtype=object)

In [40]:
#%pip install xgboost

## MODEL

In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,classification_report


In [42]:
# flll all the categorical columns at once 
le=LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col]=le.fit_transform(df[col])


In [43]:
X=df.drop('Loan_Status',axis=1)
y=df["Loan_Status"]

In [44]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#### RANDOM FOREST

In [45]:
rf=RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)
print("RF ACCURACY: ",accuracy_score(y_test,y_pred))


RF ACCURACY:  0.7225


#### GRADIENT BOOSTING

In [46]:
gb=GradientBoostingClassifier(random_state=42)

gb.fit(X_train,y_train)

y_pred=gb.predict(X_test)

print("GB Accuracy: ",accuracy_score(y_test,y_pred))

GB Accuracy:  0.7275


#### ADABOOST

In [47]:
ab=AdaBoostClassifier(random_state=42)

ab.fit(X_train,y_train)

y_pred=ab.predict(X_test)

print("AB ACCURACY: ",accuracy_score(y_test,y_pred))

c:\Users\vaish\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AB ACCURACY:  0.71
